# Battle Generation — the geographical labels behind the K-means "battle"

**Mission** (`8_create_battle/missions.md`): prepare the *battle field* for the unsupervised
K-means experiment of the ICOCO paper (`ICOCO IEEE eXPress.pdf`). We **do not run any clustering
here** — this notebook only creates the **ground-truth geographic tags** reserved for the post-hoc
evaluation of the clusters.

**Paper context.** The paper clusters 50,510 recipes (30,621 geographically labeled, 19,889
unlabeled) *jointly*; the geographic origin tags are kept **solely for post-hoc evaluation**
(Partial-Match Accuracy / Soft-Macro-F1 through Hungarian bipartite matching at K = 6 / 31 / 207).

**What this notebook creates** (as DuckDB tables instead of loose legacy files):

| Legacy file (`old_code/15_KMean_battle_field/`) | New DuckDB table | Content |
|---|---|---|
| `09_gold_data_create/data/gold_geography.csv` | `battle.cuisine_gold` | geography gold per raw cuisine tag (majority vote, >= 2 of 3 replicas) |
| `data/MasterBattle.csv` | `battle.master_battle` | per-dish geography tags (countries / sub-regions / continents) |
| `battle/indices_has_cuisine.txt` + `battle/indices_no_cuisine.txt` | `battle.recipe_status` | per-dish `id` + label status (`has_geo_label` / `no_geo_label`) |

**Row alignment with the vector files.** The `.npy` vector files built in `7_create_npy_files`
(`recipe_geography_vectors.npy`, `entity_vectors.npy`, `recipe_odor_vectors.npy`,
`recipe_odor_entropy.npy`) are ordered by `silver.RecipeZenodo` `rowid`. The `battle` `id` is
exactly `rowid + 1`, so `npy_row = battle_id - 1` selects the right recipe's vector.


# 0. Setup connection with DuckDB


## Connect to the DuckDB


In [1]:
%load_ext sql
%config SqlMagic.displaylimit = 100  # show all rows in query results
%sql duckdb:////workspace/ICOCO.duckdb


Connecting to 'duckdb:////workspace/ICOCO.duckdb'

## Close connection to DuckDB


In [2]:
# Close the connection FIRST to flush the WAL file (ICOCO.duckdb.wal)
# and avoid data corruption from an aborted transaction.
%sql --close duckdb:////workspace/ICOCO.duckdb


## Reconnect with a fresh connection


In [3]:
# Reconnect with a fresh connection (recovers/checkpoints the WAL).
%sql duckdb:////workspace/ICOCO.duckdb


Connecting and switching to connection 'duckdb:////workspace/ICOCO.duckdb'

## Setup constraint


In [4]:
from sql.connection import ConnectionManager

# IMPORTANT: DuckDB allows only ONE connection to the database at a time.
# Reuse the SAME connection that %sql already opened (do NOT duckdb.connect() again).
raw = ConnectionManager().current.connection_sqlalchemy.connection

# Keep RAM safe inside the 3.68 GB container: cap DuckDB memory + threads.
raw.execute("SET memory_limit = '1.5GB'")
raw.execute("SET threads = 4")
print('Reusing the single %sql connection for Python-side fetches (raw):', raw)


Reusing the single %sql connection for Python-side fetches (raw): <sqlalchemy.pool.base._ConnectionFairy object at 0x7211a8b0a4a0>


# 1. Check whether the standardized data exists

The battle needs the **cuisine normalization** output — the LLM-standardized geography of the raw
`Cuisine` tags stored in `silver.RecipeZenodo`.

- In this database the 3 LLM replicas live in the `cuisine_normalize` schema as
  `output1` / `output2` / `output3` (116 rows each = one row per raw cuisine tag).
- There is **no gold table yet**, so we build `battle.cuisine_gold` here.
- The legacy pipeline produced its gold with a **>= 2-of-3 majority vote** per field
  (*investigation report: "old record-level match >=2/3 ranged 96-98% per field"*).
  We apply the exact same rule, and we ignore the degenerate `World` edge case (all three
  LLM answers were truncated, so it receives no votes and drops out naturally).


In [5]:
%%sql
-- (a) Which cuisine tables already exist in this database?
--      The 3 LLM replicas live in the `cuisine_normalize` schema (schema = task folder),
--      as `output1` / `output2` / `output3`.
SELECT table_schema, table_name
FROM information_schema.tables
WHERE table_schema ILIKE '%cuisine%'
   OR table_name  ILIKE '%cuisine%'
ORDER BY 1, 2;

-- (b) Is there already a GOLD version of the cuisine data?
--      -> 0 means "no", so we create `battle.cuisine_gold` in the next cell.
SELECT COUNT(*) AS cuisine_gold_tables
FROM information_schema.tables
WHERE table_name ILIKE '%cuisine_gold%';


Running query in 'duckdb:////workspace/ICOCO.duckdb'

cuisine_gold_tables
1


In [6]:
%%sql
-- Build the GOLD geography vocabulary from the 3 LLM replicas (`cuisine_normalize.output1/2/3`).
-- Rule (identical to the legacy `old_code/09_gold_data_create/data/gold_geography.csv`,
-- cf. `investigate_data_mismatch/investigation_report.md`:
--      "old record-level match >= 2/3 ranged 96-98% per field"):
--      an item is kept iff it appears in AT LEAST 2 of the 3 replicas (>= 2/3 majority).
-- The degenerate edge-case tag "World" receives NO votes at all (all 3 answers were
-- truncated / unparsable), so it is naturally excluded.
CREATE SCHEMA IF NOT EXISTS battle;

CREATE OR REPLACE TABLE battle.cuisine_gold AS
WITH all_runs AS (
    SELECT custom_id, input_text, country_list, sub_region_list, continent_list
    FROM cuisine_normalize.output1
    UNION ALL
    SELECT custom_id, input_text, country_list, sub_region_list, continent_list
    FROM cuisine_normalize.output2
    UNION ALL
    SELECT custom_id, input_text, country_list, sub_region_list, continent_list
    FROM cuisine_normalize.output3
),
-- one row per (tag, item) across all 3 runs ----------------------------------
country_votes AS (
    SELECT custom_id, input_text, item, COUNT(*) AS n_votes
    FROM (SELECT custom_id, input_text, UNNEST(country_list) AS item FROM all_runs)
    GROUP BY 1, 2, 3
),
sub_region_votes AS (
    SELECT custom_id, input_text, item, COUNT(*) AS n_votes
    FROM (SELECT custom_id, input_text, UNNEST(sub_region_list) AS item FROM all_runs)
    GROUP BY 1, 2, 3
),
continent_votes AS (
    SELECT custom_id, input_text, item, COUNT(*) AS n_votes
    FROM (SELECT custom_id, input_text, UNNEST(continent_list) AS item FROM all_runs)
    GROUP BY 1, 2, 3
),
-- keep only items that at least 2 of the 3 replicas agree on ------------------
country_gold AS (
    SELECT custom_id, input_text, list(item ORDER BY item) AS final_country_list
    FROM country_votes WHERE n_votes >= 2 GROUP BY 1, 2
),
sub_region_gold AS (
    SELECT custom_id, input_text, list(item ORDER BY item) AS final_sub_regions
    FROM sub_region_votes WHERE n_votes >= 2 GROUP BY 1, 2
),
continent_gold AS (
    SELECT custom_id, input_text, list(item ORDER BY item) AS final_continents
    FROM continent_votes WHERE n_votes >= 2 GROUP BY 1, 2
),
-- one row per raw cuisine tag -------------------------------------------------
gold AS (
    SELECT COALESCE(a.custom_id, b.custom_id, c.custom_id)    AS custom_id,
           COALESCE(a.input_text, b.input_text, c.input_text) AS input_text,
           COALESCE(a.final_country_list, []) AS final_country_list,
           COALESCE(b.final_sub_regions, [])  AS final_sub_regions,
           COALESCE(c.final_continents, [])   AS final_continents
    FROM country_gold   a
    FULL OUTER JOIN sub_region_gold b USING (custom_id)
    FULL OUTER JOIN continent_gold  c USING (custom_id)
)
-- drop the fully-degenerate tags (every list empty), e.g. "World" -------------
SELECT custom_id, input_text, final_country_list, final_sub_regions, final_continents
FROM gold
WHERE len(final_country_list) + len(final_sub_regions) + len(final_continents) > 0;


Running query in 'duckdb:////workspace/ICOCO.duckdb'

Count


In [7]:
%%sql
-- Sanity check: the new gold must match the legacy `gold_geography.jsonl` entry-per-entry.
-- (Compared as SORTED sets: the legacy lists keep first-appearance order, while we order
-- alphabetically, so only the sets matter.)
WITH legacy AS (
    SELECT custom_id,
           list_sort(list_distinct(final_country_list)) AS cl,
           list_sort(list_distinct(final_sub_regions))  AS sl,
           list_sort(list_distinct(final_continents))   AS kk
    FROM read_json('/workspace/data/cuisine_normalize/gold_geography.jsonl')
),
newest AS (
    SELECT custom_id,
           list_sort(final_country_list) AS cl,
           list_sort(final_sub_regions)  AS sl,
           list_sort(final_continents)   AS kk
    FROM battle.cuisine_gold
)
SELECT (SELECT COUNT(*) FROM legacy)                          AS legacy_gold_tags,
       (SELECT COUNT(*) FROM newest)                          AS new_gold_tags,
       (SELECT COUNT(*) FROM legacy l
              FULL OUTER JOIN newest n USING (custom_id)
        WHERE (l.cl = n.cl) IS NOT TRUE
           OR (l.sl = n.sl) IS NOT TRUE
           OR (l.kk = n.kk) IS NOT TRUE)                      AS rows_differing;


Running query in 'duckdb:////workspace/ICOCO.duckdb'

legacy_gold_tags,new_gold_tags,rows_differing
109,109,0


# 2. Brief report

The numbers behind the battle, straight from the tables we just built.


In [8]:
%%sql
-- (a) How many normalized geography values after the majority vote?
--     Expect: 109 gold tags (the same 109 tags as the legacy gold).
SELECT COUNT(*)                                            AS gold_tags,
       COUNT(*) FILTER (WHERE len(final_country_list) > 0) AS tags_with_countries
FROM battle.cuisine_gold;


Running query in 'duckdb:////workspace/ICOCO.duckdb'

gold_tags,tags_with_countries
109,107


In [9]:
%%sql
-- (b) How many distinct geography TERMS does the gold use?
--     Expect: 207 countries / 31 sub-regions / 6 continents
--     (these are the exact numbers reported in the paper, Table I).
SELECT 'countries'   AS granularity, COUNT(DISTINCT c) AS n_terms
FROM battle.cuisine_gold, UNNEST(final_country_list) AS c
UNION ALL
SELECT 'sub_regions', COUNT(DISTINCT s) FROM battle.cuisine_gold, UNNEST(final_sub_regions) AS s
UNION ALL
SELECT 'continents',  COUNT(DISTINCT k) FROM battle.cuisine_gold, UNNEST(final_continents) AS k;


Running query in 'duckdb:////workspace/ICOCO.duckdb'

granularity,n_terms
countries,207
sub_regions,31
continents,6


In [10]:
%%sql
-- (b) How many dishes carry a cuisine tag, and how many do not?
--     Expect: ~31,585 dishes with at least one tag, ~18,926 without.
SELECT COUNT(*)                                 AS total_recipes,
       COUNT(*) FILTER (WHERE len(Cuisine) > 0) AS recipes_with_tag,
       COUNT(*) FILTER (WHERE len(Cuisine) = 0) AS recipes_without_tag
FROM silver.RecipeZenodo;


Running query in 'duckdb:////workspace/ICOCO.duckdb'

total_recipes,recipes_with_tag,recipes_without_tag
50511,31585,18926


In [11]:
%%sql
-- (c) How many distinct RAW cuisine tags exist at the element level?
--     Expect: ~116 raw tags (the input vocabulary of `cuisine_normalize`).
SELECT COUNT(DISTINCT tag) AS distinct_raw_cuisine_tags
FROM silver.RecipeZenodo, UNNEST(Cuisine) AS t(tag);


Running query in 'duckdb:////workspace/ICOCO.duckdb'

distinct_raw_cuisine_tags
116


In [12]:
%%sql
-- (d) How many dishes HAVE a label and how many DO NOT?
--     A dish is "geographically labeled" iff at least ONE of its cuisine tags
--     resolves to >= 1 country in the gold (>= 2/3 majority).
--     Expect: 50,511 total; 30,621 labeled; 19,890 unlabeled.
--     (Paper: 30,621 / 19,889 on 50,510 - the +-1 rows come from CSV parsing artifacts.)
WITH base AS (
    SELECT row_number() OVER (ORDER BY rowid) AS id, Name, Cuisine
    FROM silver.RecipeZenodo
),
joined AS (
    SELECT b.id, b.Name, g.final_country_list
    FROM base b
    LEFT JOIN UNNEST(b.Cuisine) AS t(tag) ON true
    LEFT JOIN battle.cuisine_gold g ON lower(g.input_text) = lower(trim(t.tag))
),
per_dish AS (
    SELECT id, Name,
           list_distinct(flatten(list(COALESCE(final_country_list, [])))) AS final_country_list
    FROM joined GROUP BY id, Name
)
SELECT COUNT(*)                                       AS total_recipes,
       COUNT(*) FILTER (WHERE len(final_country_list) > 0) AS has_geo_label,
       COUNT(*) FILTER (WHERE len(final_country_list) = 0) AS no_geo_label
FROM per_dish;


Running query in 'duckdb:////workspace/ICOCO.duckdb'

total_recipes,has_geo_label,no_geo_label
50511,30621,19890


# 3. Define the "label" for posthoc evaluation

Each dish already has a `rowid` in `silver.RecipeZenodo`, and the vector `.npy` files follow that
order. We assign a stable **1-based `id` = `rowid + 1`** and attach the gold geography tags,
reproducing `old_code/15_KMean_battle_field/data/MasterBattle.csv` inside DuckDB.


In [13]:
%%sql
-- Rebuild the legacy `old_code/15_KMean_battle_field/data/MasterBattle.csv` as a DuckDB table.
-- The 1-based `id` is the position of the recipe in `silver.RecipeZenodo` (rowid order),
-- so it matches the row order of the vector `.npy` files created in `7_create_npy_files`:
--         npy_row_index = id - 1
CREATE OR REPLACE TABLE battle.master_battle AS
WITH base AS (
    SELECT row_number() OVER (ORDER BY rowid) AS id, Name, Cuisine
    FROM silver.RecipeZenodo
),
joined AS (
    SELECT b.id, b.Name,
           g.final_country_list, g.final_sub_regions, g.final_continents
    FROM base b
    LEFT JOIN UNNEST(b.Cuisine) AS t(tag) ON true
    LEFT JOIN battle.cuisine_gold g ON lower(g.input_text) = lower(trim(t.tag))
),
agg AS (
    SELECT id, Name,
           list_distinct(flatten(list(COALESCE(final_country_list, [])))) AS final_country_list,
           list_distinct(flatten(list(COALESCE(final_sub_regions, []))))  AS final_sub_regions,
           list_distinct(flatten(list(COALESCE(final_continents, []))))   AS final_continents
    FROM joined
    GROUP BY id, Name
)
SELECT id, Name, final_country_list, final_sub_regions, final_continents,
       len(final_country_list) AS country_count,
       len(final_sub_regions)  AS sub_region_count,
       len(final_continents)   AS continent_count
FROM agg
ORDER BY id;


Running query in 'duckdb:////workspace/ICOCO.duckdb'

Count


In [14]:
%%sql
-- Verification: same split as the paper (30,621 labeled / 19,890 unlabeled).
SELECT COUNT(*)                                      AS total_recipes,
       COUNT(*) FILTER (WHERE country_count > 0)     AS has_geo_label,
       COUNT(*) FILTER (WHERE country_count = 0)     AS no_geo_label
FROM battle.master_battle;


Running query in 'duckdb:////workspace/ICOCO.duckdb'

total_recipes,has_geo_label,no_geo_label
50511,30621,19890


In [15]:
%%sql
-- Spot-check the first rows (must read like the legacy MasterBattle.csv).
SELECT * FROM battle.master_battle ORDER BY id LIMIT 5;


Running query in 'duckdb:////workspace/ICOCO.duckdb'

id,Name,final_country_list,final_sub_regions,final_continents,country_count,sub_region_count,continent_count
1,Pineapple Glaze for Ham,['United States of America'],['Northern America'],['North America'],1,1,1
2,Awesome Egg Rolls,"['Yemen', 'United Arab Emirates', 'Turkmenistan', 'Tajikistan', 'South Korea', 'Singapore', 'Saudi Arabia', 'Russia', 'Qatar', 'Syria', 'Philippines', 'Palestine', 'Pakistan', 'Oman', 'North Korea', 'Georgia', 'China', 'India', 'Cyprus', 'Malaysia', 'Taiwan', 'East Timor', 'Lebanon', 'Azerbaijan', 'Bangladesh', 'Jordan', 'Bahrain', 'Uzbekistan', 'Cambodia', 'Armenia', 'Sri Lanka', 'Bhutan', 'Nepal', 'Afghanistan', 'Iran', 'Kuwait', 'Vietnam', 'Brunei', 'Maldives', 'Indonesia', 'Israel', 'Myanmar', 'Iraq', 'Japan', 'Turkey', 'Kazakhstan', 'Kyrgyzstan', 'Laos', 'Thailand', 'Mongolia']","['Western Asia', 'Southern Asia', 'South-eastern Asia', 'Eastern Asia', 'Central Asia']",['Asia'],50,5,1
3,Spanish Tortilla,['Spain'],['Southern Europe'],['Europe'],1,1,1
4,Spicy Tarragon Yogurt Chicken,['United States of America'],['Northern America'],['North America'],1,1,1
5,Apple Bread,['United States of America'],['Northern America'],['North America'],1,1,1


# 4. Export the id of the dish with label and unlabel

A single `(id, status)` table that merges the legacy `indices_has_cuisine.txt` and
`indices_no_cuisine.txt`. Because the vector rows are keyed by `silver.RecipeZenodo.rowid`,
`id = rowid + 1` is also the key to slice the `.npy` matrices (vector row = `id - 1`).


In [16]:
%%sql
-- Merge the two legacy files
--     old_code/15_KMean_battle_field/battle/indices_has_cuisine.txt
--     old_code/15_KMean_battle_field/battle/indices_no_cuisine.txt
-- into a single `(id, status)` table.
--   status = 'has_geo_label' -> dish maps to >= 1 country (labels available for post-hoc eval)
--   status = 'no_geo_label'  -> dish has no usable country labels
CREATE OR REPLACE TABLE battle.recipe_status AS
SELECT id,
       CASE WHEN country_count > 0 THEN 'has_geo_label' ELSE 'no_geo_label' END AS status
FROM battle.master_battle
ORDER BY id;


Running query in 'duckdb:////workspace/ICOCO.duckdb'

Count


In [17]:
%%sql
-- Expect: 30,621 `has_geo_label` + 19,890 `no_geo_label` = 50,511 rows.
SELECT status, COUNT(*) AS n_recipes
FROM battle.recipe_status
GROUP BY status
ORDER BY status;


Running query in 'duckdb:////workspace/ICOCO.duckdb'

status,n_recipes
has_geo_label,30621
no_geo_label,19890


In [18]:
%%sql
-- The ids must be exactly 1 .. 50,511 (no gaps, no duplicates).
SELECT MIN(id) AS first_id, MAX(id) AS last_id,
       COUNT(*) AS n_rows, COUNT(DISTINCT id) AS n_distinct_ids
FROM battle.recipe_status;


Running query in 'duckdb:////workspace/ICOCO.duckdb'

first_id,last_id,n_rows,n_distinct_ids
1,50511,50511,50511


In [19]:
%%sql
-- A peek at both groups (like the mission's example rows).
SELECT 'has_geo_label' AS status, id FROM battle.recipe_status
WHERE status = 'has_geo_label' ORDER BY id LIMIT 3;


Running query in 'duckdb:////workspace/ICOCO.duckdb'

status,id
has_geo_label,1
has_geo_label,2
has_geo_label,3


In [20]:
%%sql
-- A peek at both groups (like the mission's example rows).
SELECT 'no_geo_label' AS status, id FROM battle.recipe_status
WHERE status = 'no_geo_label' ORDER BY id LIMIT 3;


Running query in 'duckdb:////workspace/ICOCO.duckdb'

status,id
no_geo_label,31
no_geo_label,239
no_geo_label,257


# 5. Summary — the battle field is ready

Everything is idempotent (`CREATE OR REPLACE`), so re-running this notebook rebuilds the same
tables without touching anything else in the database.

| Table (schema `battle`) | Rows | Meaning |
|---|---|---|
| `cuisine_gold` | 109 | geography gold per raw cuisine tag (>= 2/3 majority) — 207 countries / 31 sub-regions / 6 continents |
| `master_battle` | 50,511 | per-dish labels: `id`, `Name`, country / sub-region / continent lists (+ counts) |
| `recipe_status` | 50,511 | per-dish `(id, status)` — 30,621 `has_geo_label`, 19,890 `no_geo_label` |

**Slicing the vector matrices for a battle** (the actual clustering is the NEXT notebook — not this one):

```python
import numpy as np
has_idx = recipe_status[recipe_status.status == 'has_geo_label'].id.to_numpy() - 1   # npy rows
no_idx  = recipe_status[recipe_status.status == 'no_geo_label'].id.to_numpy() - 1
```

| Battle subset | Rows in the `.npy` matrices (`id - 1`) |
|---|---|
| Full dataset | `0 … 50,510` (all recipes) |
| Labeled subset | ids with `status = 'has_geo_label'` (30,621) |
| Unlabeled subset | ids with `status = 'no_geo_label'` (19,890) |

For the external-alignment metrics, the clusters are mapped to ground truth via Hungarian matching
at the country / sub-region / continent granularities (K = 207 / 31 / 6), using respectively
`master_battle.final_country_list` / `final_sub_regions` / `final_continents`.


In [21]:
# Close the connection FIRST to flush the WAL file (ICOCO.duckdb.wal)
# and avoid data corruption from an aborted transaction.
%sql --close duckdb:////workspace/ICOCO.duckdb
